# AfriSpeech DataGen — synthetic TTS data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfriSpeech/afrispeech-datagen/blob/main/notebooks/afrispeech_datagen.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/AfriSpeech/afrispeech-datagen/blob/main/notebooks/afrispeech_datagen.ipynb)

Generate synthetic TTS training audio from text, voice-cloning the built-in
male/female speakers. Output: `wavs/*.wav` + a manifest in your TTS format
(LJSpeech / Piper / VITS / MeloTTS) — ready for your trainer.

> **Use a GPU runtime** (required). Colab: `Runtime → Change runtime type → T4 GPU`.
> Kaggle: `Settings → Accelerator → GPU` **and Internet ON**.

## Install

In [ ]:
!apt-get -qq install -y ffmpeg >/dev/null
!git clone -q https://github.com/AfriSpeech/afrispeech-datagen.git
%cd afrispeech-datagen
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — switch to a GPU runtime!'

## (Optional) Hugging Face token
Needed if the model or your dataset is gated/private. Skip otherwise.

In [ ]:
import os, getpass
_tok = getpass.getpass('HF token (optional — Enter to skip): ').strip()
if _tok:
    os.environ['HF_TOKEN'] = _tok
    print('Token set.')
else:
    print('No token set.')

## Configure your run
Point at a text dataset on the Hub and its text column. (List AfriSpeech-org
datasets with `!python -m afrispeech_datagen --list-datasets`.)

In [ ]:
DATASET = "ghananlpcommunity/CHANGE-ME"   # your text dataset id (or use a text file below)
TEXT_COLUMN = "text"                        # the text column
# TEXT_FILE = "sentences.txt"               # alternative source: one sentence per line
NAME = "demo"                                # output → data/demo/
MAX_SAMPLES = 20                             # use at most N input rows (None = all; capped by --hours too)
FORMATS = "ljspeech"                         # ljspeech | piper | vits | melo (comma list)
SR = 22050                                   # output sample rate (e.g. 24000 for MeloTTS)
PRECISION = "fp32"                           # fp16 = faster/½ VRAM on a T4 (check quality); bf16 needs A100/L4

## Preview a few clips
Hear the result before a big run.

In [ ]:
!python -m afrispeech_datagen --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --name {NAME} --sample-rate {SR} --precision {PRECISION} --preview 3

import glob
from IPython.display import Audio, display
for w in sorted(glob.glob(f'data/{NAME}/preview/*.wav')):
    print(w)
    display(Audio(w))

## Generate
Small demo run (0.2 h) — bump `--hours` for a real run. Re-run the same cell to
**resume** (finished rows are skipped).

**Speed on a GPU:** instances auto-size to your VRAM (≈2 on a T4). Push a T4 with
`--instances 3`, or `--precision fp16` for ~2× speed / half VRAM (preview the
quality first). `--voices male|female` and `--male-pct N` control the speakers.

In [ ]:
!python -m afrispeech_datagen --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --max-samples {MAX_SAMPLES} --name {NAME} --formats {FORMATS} \
    --sample-rate {SR} --precision {PRECISION}
# tip: add  --instances 3  (push a T4) or  --precision fp16  (faster) once quality looks good
# control size with  --max-samples N  and/or  --hours H  (whichever is reached first)
# from your own sentences instead:  --text-file {TEXT_FILE}  (drop --dataset/--text-column)

## Inspect the output

In [ ]:
import json, glob
from IPython.display import Audio, display

rows = [json.loads(l) for l in open(f'data/{NAME}/manifest.jsonl', encoding='utf-8')]
print(f'{len(rows)} clips, {sum(r["duration"] for r in rows)/3600:.3f} h')
for r in rows[:3]:
    print(f"\n[{r['gender']}] {r['duration']}s  {r['text'][:90]}")
    display(Audio(f"data/{NAME}/{r['file']}"))

## (Optional) Push to your own HF dataset repo
```bash
!python -m afrispeech_datagen --dataset {DATASET} --text-column {TEXT_COLUMN} \
    --hours 0.2 --name {NAME} --formats {FORMATS} --sample-rate {SR} \
    --push your-username/my-synth --token $HF_TOKEN
```
Full options: https://github.com/AfriSpeech/afrispeech-datagen